# DB에서 상장 기업 정보를 받아 네이버 증권에서 주가 수집하기

In [104]:
import os
import time
import pandas as pd
from datetime import time
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pymysql
import requests
from bs4 import BeautifulSoup as bs

In [11]:
pymysql.install_as_MySQLdb()
load_dotenv(dotenv_path=".env_db")

True

In [12]:
# mysql에서 테이블 불러오기
engine = create_engine(f"{os.getenv('db')}+{os.getenv('dbtype')}://{os.getenv('id')}:{os.getenv('pw')}@{os.getenv('host')}/{os.getenv('database')}")
conn = engine.connect()
data = pd.read_sql('2024_07_29_stock_company_info2', con=conn)
conn.close()

In [133]:
def str2int(x):
    x = int(x.replace(',', ''))
    return x

In [136]:
stock_price_detail = {}
for idx, code in enumerate(data['종목코드'][:10]):
    url = f"https://finance.naver.com/item/main.naver?code={code}"
    r = requests.get(url)
    print(r.status_code, f"{idx+1}/{len(data['종목코드'])} 수집중", end='\r' )
    soup = bs(r.text, 'lxml')
    price = int((soup.select_one(".today").text).strip('\n').split('\n')[1].replace(',', ""))
    price_change = int((soup.select_one(".today").text).strip('\n').split('\n')[9].replace(',', ""))
    rate_change = float(((soup.select_one(".today").text).strip('\n').split('\n')[13]+(soup.select_one(".today").text).strip('\n').split("\n")[15]).replace("%",""))
    stock_price_detail.setdefault('현재가', []).append(price)
    stock_price_detail.setdefault('변동금액', []).append(price_change)
    stock_price_detail.setdefault('변화율', []).append(rate_change)
    
    table = soup.select_one(".no_info")
    for tr in table.select("tr"):
        for td in tr.select("td"):
            stock_price_detail.setdefault(td.select_one("span").text, []).append(str2int(td.select_one("span.blind").text))
    time.sleep(5)
stock_price_detail

ValueError: could not convert string to float: '+\n'

In [107]:
soup.select_one(".rate_info")
soup

<html lang="ko">
<head>
<title>산일전기 : 네이버페이 증권</title>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="text/javascript" http-equiv="Content-Script-Type"/>
<meta content="text/css" http-equiv="Content-Style-Type"/>
<meta content="네이버페이 증권" name="apple-mobile-web-app-title"/>
<meta content="https://finance.naver.com/item/main.naver?code=062040" property="og:url"/>
<meta content="산일전기 - 네이버페이 증권 : 네이버페이 증권" property="og:title"/>
<meta content="관심종목의 실시간 주가를 가장 빠르게 확인하는 곳" property="og:description"/>
<meta content="https://ssl.pstatic.net/static/m/stock/im/2016/08/og_stock-200.png" property="og:image"/>
<meta content="article" property="og:type"/>
<meta content="" property="og:article:thumbnailUrl"/>
<meta content="네이버페이 증권" property="og:article:author"/>
<meta content="http://FINANCE.NAVER.COM" property="og:article:author:url"/>
<link href="https://ssl.pstatic.net/imgstock/static.pc/20240725112316/css/finance_header.css" rel="stylesheet" type="text/css"

In [115]:
price = int((soup.select_one(".today").text).strip('\n').split('\n')[1].replace(',', ""))
price

50200

In [117]:
price_change = int((soup.select_one(".today").text).strip('\n').split('\n')[9].replace(',', ""))
price_change

15200

In [116]:
rate_change = int((soup.select_one(".today").text).strip('\n').split('\n')[13].replace(',', ""))
rate_change

ValueError: invalid literal for int() with base 10: '+'

In [121]:
table = soup.select_one(".no_info")

In [128]:
(table.select("tr")[0].select_one("td span.blind").text)

'35,000'

In [127]:
(table.select("tr")[0].text).strip('\n').replace('\n\n', '\n')

'전일\n35,000\n35,000\n\n고가\n50,20050,200\n(상한가\n140,000140,000\n)\n\n거래량\n17,066,335\n17,066,336'

In [ ]:
(table.select("tr")[0].text).strip('\n').replace('\n\n', '\n')

In [132]:
stock_price_detail = {}
for tr in table.select("tr"):
    for td in tr.select("td"):
        stock_price_detail.setdefault(td.select_one("span").text, []).append(str2int(td.select_one("span.blind").text))
stock_price_detail

{'전일': [35000],
 '고가': [50200],
 '거래량': [17066335],
 '시가': [44900],
 '저가': [40200],
 '거래대금': [761358]}

In [17]:
data

,증권종류,회사명,종목코드,업종,주요제품,상장일,결산월,대표자명,홈페이지,지역
0,유가증권,산일전기,062040,"전동기, 발전기 및 전기 변환 · 공급 · 제어 장치 제조업","유입, 몰드, 주상, 건식 변압기 등",2024-07-29,12월,박동석,http://www.sanil.co.kr,경기도
1,유가증권,에이치에스효성,487570,기타 금융업,지주사업,2024-07-29,12월,조현상..,http://www.hshyosung.com,서울특별시
2,"코스닥, 투자주의종목",엔에이치스팩31호,481890,금융 지원 서비스업,금융지원서비스업,2024-07-26,12월,이시형,,서울특별시
3,코스닥,SK증권제13호스팩,473950,금융 지원 서비스업,기업인수목적 주식회사,2024-07-25,12월,임율표,,서울특별시
4,코스닥,엑셀세라퓨틱스,373110,기초 의약물질 제조업,CellCor SFD/CD(세포배양배지),2024-07-15,12월,이의일,,서울특별시
...,...,...,...,...,...,...,...,...,...,...
2730,"유가증권, KTOP30, KOSPI200, KRX300",유한양행,000100,의약품 제조업,"의약품(삐콤씨, 안티푸라민, 렉라자, 로수바미브, 코푸시럽 등), 생활용품(유한락스...",1962-11-01,12월,대표이..,http://www.yuhan.co.kr,서울특별시
2731,"유가증권, KOSPI200, KRX300",CJ대한통운,000120,도로 화물 운송업,"Contract Logistics, 포워딩, 항만하역, 해운, 택배국제특송, SCM...",1956-07-02,12월,신영수..,http://www.cjlogistics.com,서울특별시
2732,유가증권,경방,000050,종합 소매업,"섬유류(면사,면혼방사,면직물,면혼방직물,화섬사,화섬직물) 제조,도매,수출입",1956-03-03,12월,"김준,..",http://www.kyungbang.co.kr,서울특별시
2733,유가증권,유수홀딩스,000700,회사 본부 및 경영 컨설팅 서비스업,지주사업,1956-03-03,12월,송영규,http://www.eusu-holdings.com,서울특별시


In [30]:
url = f"https://finance.naver.com/item/main.naver?code={code}"
r = requests.get(url)
print(r.status_code, end="\r")
soup = bs(r.text, 'lxml')
soup

<html lang="ko">
<head>
<title>산일전기 : 네이버페이 증권</title>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="text/javascript" http-equiv="Content-Script-Type"/>
<meta content="text/css" http-equiv="Content-Style-Type"/>
<meta content="네이버페이 증권" name="apple-mobile-web-app-title"/>
<meta content="https://finance.naver.com/item/main.naver?code=062040" property="og:url"/>
<meta content="산일전기 - 네이버페이 증권 : 네이버페이 증권" property="og:title"/>
<meta content="관심종목의 실시간 주가를 가장 빠르게 확인하는 곳" property="og:description"/>
<meta content="https://ssl.pstatic.net/static/m/stock/im/2016/08/og_stock-200.png" property="og:image"/>
<meta content="article" property="og:type"/>
<meta content="" property="og:article:thumbnailUrl"/>
<meta content="네이버페이 증권" property="og:article:author"/>
<meta content="http://FINANCE.NAVER.COM" property="og:article:author:url"/>
<link href="https://ssl.pstatic.net/imgstock/static.pc/20240725112316/css/finance_header.css" rel="stylesheet" type="text/css"

In [62]:
result = soup.select_one("dl.blind > :nth-of-type(4)").string
cur_price = result.split(' ')[1].replace(',', '')
cur_price

'46750'

In [61]:
incr_price = result.split(' ')[4].replace(',', '')
incr_price

'11750'

In [60]:
incr_per = result.split(' ')[6].replace(',', '')
incr_per

'33.57'

In [59]:
result2 = soup.select_one("dl.blind > :nth-of-type(5)").string
yes_price = result2.split(' ')[1].replace(',', '')
yes_price

'35000'

In [64]:
result3 = soup.select_one("dl.blind > :nth-of-type(6)").string
open_price = result3.split(' ')[1].replace(',', '')
open_price

'44900'

In [65]:
result4 = soup.select_one("dl.blind > :nth-of-type(7)").string
high_price = result4.split(' ')[1].replace(',', '')
high_price

'49200'

In [67]:
result5 = soup.select_one("dl.blind > :nth-of-type(8)").string
upplm_price = result5.split(' ')[1].replace(',', '')
upplm_price

'140000'

In [68]:
result6 = soup.select_one("dl.blind > :nth-of-type(9)").string
low_price = result6.split(' ')[1].replace(',', '')
low_price

'40200'

In [70]:
result7 = soup.select_one("dl.blind > :nth-of-type(10)").string
lowlm_price = result7.split(' ')[1].replace(',', '')
lowlm_price

'21000'

In [71]:
result8 = soup.select_one("dl.blind > :nth-of-type(11)").string
trad_vol = result8.split(' ')[1].replace(',', '')
trad_vol

'15286703'

In [78]:
result9 = soup.select_one("dl.blind > :nth-of-type(12)").string
trad_val = result9.split(' ')[1].replace(',', '')
trad_val = trad_val.replace('백만', '000000')
trad_val

'676940000000'

In [ ]:
stock_price_info = soup.select_one("dl.blind

In [ ]:
page = 1
page_size = 100
final_result_df = pd.DataFrame()

while True:
    for code in data['종목코드'][0]:
        url = f"https://finance.naver.com/item/main.naver?code={code}"
        r = requests.get(url)
        soup = bs(r.text, 'lxml')
        
        for idx, (key, td) in enumerate(zip(keys, tr.select('td'))):
        
        
        
        time.sleep(5)
        

In [101]:
keys = soup.select("dl.blind > dd").string

# keys[3], keys[4:12]
# for key in keys:
    

AttributeError: ResultSet object has no attribute 'string'. You're probably treating a list of elements like a single element. Did you call find_all() when you meant to call find()?

# 수집과 동시에 DB에 저장하기 + 예외 처리하기

In [ ]:
def stock_info_to_db(date, idx, code, df):
    from sqlalchemy import create_engine
    import pymysql
    import pandas as pd
    from dotenv import load_dotenv
    from datetime import date
    pymysql.install_as_MySQLdb()
    load_dotenv(dotenv_path=".env_db")
    today = str(date, today()).replace('-', '_')
    
    engine = create_engine(f"{os.getenv('db')}+{os.getenv('dbtype')}://{os.getenv('id')}:{os.getenv('pw')}@{os.getenv('host')}/{os.getenv('database')}")
    conn = engine.connect()
    df.to_sql(f'{today}_stock_price_info', con=conn, if exists='append', index=False)
    
    return print(f'{today}, {inx}, {code}, {'저장완료':<30s}', end='\r')

In [ ]:
errors = {}
for idx, code in enumerate(zip(data['회사명'], data['종목코드'][:10]):
    stock_price_detail = {}
    url = f"https://finance.naver.com/item/main.naver?code={code}"
    try:
        r = requests.get(url)
        print(r.status_code, f"{idx+1}/{len(data['종목코드'])} 수집중                   ", end='\r' )
        soup = bs(r.text, 'lxml')
        price = int((soup.select_one(".today").text).strip('\n').split('\n')[1].replace(',', ""))
        price_change = int((soup.select_one(".today").text).strip('\n').split('\n')[9].replace(',', ""))
        rate_change = float(((soup.select_one(".today").text).strip('\n').split('\n')[13]+(soup.select_one(".today").text).strip('\n')[15]).replace('%', ''))
        stock_price_detail.setdefault('회사명', []).append(company)
        stock_price_detail.setdefault('종목코드', []).append(code)                   
        stock_price_detail.setdefault('현재가', []).append(price)
        stock_price_detail.setdefault('변동금액', []).append(price_change)
        stock_price_detail.setdefault('변화율', []).append(rate_change)

        table = soup.select_one(".no_info")
        for tr in table.select("tr"):
            for td in tr.select("td"):
                stock_price_detail.setdefault(td.select_one("span").text, []).append(str2int(td.select_one("span.blind").text))
        df = pd.DataFrame(stock_price_detail)
        stock_info_to_db(idx, code, df)
        time.sleep(5)
    except Exception as e:
        print(e)
        print(r.status_code, f"{idx+1}/{len(data['종목코드'])} 수집중", end='\r')
        errors.setdefault("에러", []).append(code)
stock_price_detail